In [0]:
%run ../01_Bronze/00_setup

In [0]:
# process new records into the Silver layer orders table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_orders(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.orders_silver"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_current = true") \
        .select("order_id")
    
    updates_df = microBatchDF.join(existing_ids, "order_id", "inner") \
        .withColumn("merge_key", F.col("order_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_view AS source
        ON target.order_id = source.merge_key AND target.is_current = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_current = false, target.end_date = current_timestamp()
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (order_sk, order_id, customer_id, orderdate, status, updated_timestamp, is_current, start_date, end_date)
          VALUES (md5(concat(source.order_id, cast(source.updated_timestamp as STRING))), 
                  source.order_id, source.customer_id, source.orderdate, source.status, 
                  current_timestamp(), true, source.updated_timestamp, NULL)
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.orders")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_orders)
    .option("checkpointLocation", f"{bronze_volume_path}/checkpoints/silver_orders_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())